# SOXX 评价段原始披露集中度逐日有效来源

固定前一训练段的最后公开报告与 14 份评价段首版 NPORT、1 份迟到修订。每个信号只选已申报且报告期最新的原始事件，完整保留 Stage1 的 944 日。0 收益、拟合和账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2023_2026_concentration_source_map_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for p,h in [(plan['training_source_summary'],plan['training_source_summary_sha256']),(plan['held_original_source_summary'],plan['held_original_source_summary_sha256']),(plan['amendment_comparison'],plan['amendment_comparison_sha256'])]:assert sha(root/p)==h
train_summary=json.loads((root/plan['training_source_summary']).read_text());held_summary=json.loads((root/plan['held_original_source_summary']).read_text());amendment=json.loads((root/plan['amendment_comparison']).read_text())
assert train_summary['base_report_events']==48 and held_summary['report_events']==15 and amendment['status']=='same_30_original_EC_rows' and amendment['changed_member_rows']==0
state={'status':'started','Qlib_source_Recorder_object_loads':0,'new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
with R.uri_context(train_summary['recorder_uri']):
 train=R.get_recorder(recorder_id=train_summary['recorder_id'],experiment_id=train_summary['experiment_id']).load_object('analysis/source_events')
state['Qlib_source_Recorder_object_loads']+=1
with R.uri_context(held_summary['recorder_uri']):
 held=R.get_recorder(recorder_id=held_summary['recorder_id'],experiment_id=held_summary['experiment_id']).load_object('analysis/source_events')
state['Qlib_source_Recorder_object_loads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
assert len(train)==48 and len(held)==15
last=train[(train.report_period=='2022-09-30')&(train.form=='NPORT-P')]
assert len(last)==1
last=last.rename(columns={'top5':'top5_disclosed_EC_value_share','HHI':'HHI_disclosed_EC_value'})
cols=['accession','report_period','filed','form','top5_disclosed_EC_value_share','HHI_disclosed_EC_value']
events=pd.concat([last[cols],held[cols]],ignore_index=True).sort_values('filed').reset_index(drop=True)
assert len(events)==16 and events.accession.is_unique and sum(events.form=='NPORT-P/A')==1
events['filed_date']=pd.to_datetime(events.filed)
calendar=pd.DatetimeIndex(D.calendar(start_time='2022-11-01',end_time='2026-10-10',freq='day'))
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
signals=calendar[(calendar>=pd.Timestamp(plan['evaluation_span'][0]))&(calendar<=pd.Timestamp(plan['evaluation_span'][1]))]
assert len(signals)==944
rows=[];ignored=0
for day in signals:
 candidates=events[events.effective_signal_date<=day]
 assert len(candidates)>0
 selected=candidates.sort_values(['report_period','filed','accession']).iloc[-1]
 older_amendment=candidates[(candidates.form=='NPORT-P/A')&(candidates.report_period<selected.report_period)]
 ignored+=len(older_amendment)>0
 rows.append({'signal_date':day,'accession':selected.accession,'report_period':selected.report_period,'filed_date':selected.filed_date,'effective_signal_date':selected.effective_signal_date,'form':selected.form,'top5':selected.top5_disclosed_EC_value_share,'HHI':selected.HHI_disclosed_EC_value,'source_age_days':(day-selected.filed_date).days,'source_known':True,'older_amendment_available_but_not_active':len(older_amendment)>0})
mapped=pd.DataFrame(rows)
assert len(mapped)==944 and mapped.signal_date.is_unique and mapped.source_known.all() and mapped.source_age_days.ge(1).all()
assert not ((mapped.signal_date>=pd.Timestamp('2026-07-14'))&(mapped.accession=='0002071691-26-015796')).any()
assert (mapped.loc[mapped.signal_date>=pd.Timestamp('2026-07-14'),'report_period']=='2026-03-31').all()
assert mapped.iloc[0].accession==last.iloc[0].accession
frame=mapped.set_index('signal_date')[['top5','HHI','source_known','source_age_days','older_amendment_available_but_not_active']]
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5','HHI']],'meta':frame[['source_known','source_age_days','older_amendment_available_but_not_active']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'Stage1_held':tuple(plan['evaluation_span'])})
prepared=dataset.prepare('Stage1_held',col_set=['feature','meta']);assert len(prepared)==944 and int(prepared[('meta','source_known')].sum())==944
audit={'status':'full_Stage1_held_original_concentration_source_map_944_dates_no_late_override','source_plan_sha256':sha(card/'plan.json'),'candidate_events':len(events),'held_base_events':sum(events.form=='NPORT-P')-1,'held_late_amendments':sum(events.form=='NPORT-P/A'),'Stage1_held_Qlib_sessions':len(signals),'source_known_sessions':int(mapped.source_known.sum()),'first_active_accession':mapped.iloc[0].accession,'last_active_accession':mapped.iloc[-1].accession,'older_amendment_available_but_not_active_sessions':int(mapped.older_amendment_available_but_not_active.sum()),'late_amendment_never_active_after_newer_report':True,'max_source_age_days':int(mapped.source_age_days.max()),'last_active_report_period':mapped.iloc[-1].report_period,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'evaluation_history_is_independent':False}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('Stage1_held',col_set=['feature','meta']))==944
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_held_original_concentration_daily_source_map',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'training':plan['training_source_summary_sha256'],'held':plan['held_original_source_summary_sha256'],'amendment':plan['amendment_comparison_sha256']})
 rec.save_objects(artifact_path='analysis',source_events=events,mapped_signals=mapped,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_events=len(events),held_sessions=944,known_source_sessions=944,older_amendment_ignored_sessions=audit['older_amendment_available_but_not_active_sessions'],model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/mapped_signals'))==944 and len(checked.load_object('metadata/dataset').prepare('Stage1_held',col_set=['feature','meta']))==944
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_held_original_concentration_daily_source_map','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-held-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'Stage1_held_original_concentration_944_source_dates_qualified_reused_history_only_training_amendment_pending','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:audit[k] for k in ('Stage1_held_Qlib_sessions','source_known_sessions','first_active_accession','last_active_accession','older_amendment_available_but_not_active_sessions','last_active_report_period','max_source_age_days')});print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'Stage1_held_Qlib_sessions': 944, 'source_known_sessions': 944, 'first_active_accession': '0001752724-22-268670', 'last_active_accession': '0002071691-26-012504', 'older_amendment_available_but_not_active_sessions': 61, 'last_active_report_period': '2026-03-31', 'max_source_age_days': 132}
{'recorder_id': 'a66747c1a1cc46b69f17fe620deadbd1', 'nas_sha256': 'dab30dc1159a2e410d5a9d9c24fd4e90ddf81f6aee0b594208363ccf0659a76f'}


## 中文结论

2022-09 的训练末报告接 2023–2026 年 14 份首版与一份迟到修订，按申报日后实际 Qlib 交易日和最新报告期选择，冻结 Stage1 评价 944/944 日均有原始披露集中度。2026-07 迟到的 2025-09 修订已可见 61 日但未覆盖 2026-03 更新报告；末日仍取 2026-03 原件。DatasetH/Recorder `a66747c1a1cc46b69f17fe620deadbd1` 与 NAS SHA-256 `dab30dc1159a2e410d5a9d9c24fd4e90ddf81f6aee0b594208363ccf0659a76f` 回读通过。0 收益读取、0 拟合、0 账户。评价历史非独立前向。